# Run the standalone ENVIRA Gradio web app
Run each cell in order. This notebook only prepares the runtime, mounts persistent storage, initializes shared resources, and launches the Python application.


In [ ]:
from getpass import getpass
from pathlib import Path
import os
import shutil
import subprocess

GITHUB_USER = "dsdengrasa08"
REPO_NAME = "ENVIRA-Phase-1-Data"

WORKSPACE_DIR = Path("/content/colab_repos")
REPO_DIR = WORKSPACE_DIR / REPO_NAME

# ------------------------------------------------------------------
# Setup workspace
# ------------------------------------------------------------------

os.chdir("/content")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace folder: {WORKSPACE_DIR}")
print(f"Target repo path: {REPO_DIR}")

# ------------------------------------------------------------------
# Check whether repository already exists
# ------------------------------------------------------------------

if REPO_DIR.is_dir() and (REPO_DIR / ".git").is_dir():
    print(f"\nRepository already exists:")
    print(f"  {REPO_DIR}")
    print("Skipping GitHub authentication and clone.")

else:
    # --------------------------------------------------------------
    # Handle incomplete/non-Git directory if one exists
    # --------------------------------------------------------------

    if REPO_DIR.exists():
        print(f"\nExisting path is not a valid Git repository:")
        print(f"  {REPO_DIR}")
        print("Removing incomplete directory before cloning...")
        shutil.rmtree(REPO_DIR)

    # --------------------------------------------------------------
    # Authentication is requested ONLY when cloning is necessary
    # --------------------------------------------------------------

    print("\nRepository not found. Cloning from GitHub...")

    token = getpass("Paste GitHub token: ")

    repo_url = (
        f"https://{GITHUB_USER}:{token}"
        f"@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    )

    result = subprocess.run(
        ["git", "clone", repo_url, str(REPO_DIR)],
        cwd=str(WORKSPACE_DIR),
        text=True,
        capture_output=True,
    )

    # Clear secrets immediately after clone attempt
    token = None
    repo_url = None

    if result.returncode != 0:
        print("Git clone failed.")
        print(result.stderr)
        raise RuntimeError("Repository was not cloned.")

    print(f"\nRepo cloned successfully to:")
    print(f"  {REPO_DIR}")

# ------------------------------------------------------------------
# ------------------------------------------------------------------

os.chdir(REPO_DIR)

print(f"\nCurrent working directory:")
print(f"  {Path.cwd()}")

In [ ]:
from pathlib import Path
import os

APP_DIR = Path.cwd().resolve()
if not (APP_DIR / "pyproject.toml").is_file():
    candidate = APP_DIR / "pdf_layout_gradio_app"
    if candidate.is_dir():
        APP_DIR = candidate.resolve()
if not (APP_DIR / "src" / "envira_layout_web").is_dir():
    raise FileNotFoundError("Open this notebook from the standalone pdf_layout_gradio_app folder.")
print("Standalone app:", APP_DIR)


In [ ]:
# Install the standalone project into this notebook kernel. Using the kernel's
# exact Python executable avoids installing into a different environment.
import importlib
import subprocess
import sys

install_target = f"{APP_DIR}[notebook]"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", install_target],
    check=True,
)

# Editable installs should add this path automatically. Add it explicitly as a
# deterministic notebook fallback (not a reference-pipeline dependency), then
# invalidate import caches before importing the application.
standalone_src = str((APP_DIR / "src").resolve())
if standalone_src not in sys.path:
    sys.path.insert(0, standalone_src)
importlib.invalidate_caches()

spec = importlib.util.find_spec("envira_layout_web")
if spec is None:
    raise ModuleNotFoundError(
        "The standalone envira_layout_web package is still unavailable after installation. "
        f"Expected it under {standalone_src}."
    )
print("Standalone package import path:", spec.origin)


In [ ]:
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_ROOT = Path("/content/drive/MyDrive/ENVIRA/pdf-layout-gradio")
else:
    # Set ENVIRA_WEB_OUTPUT_ROOT to an already-mounted persistent location.
    OUTPUT_ROOT = Path(os.environ.get("ENVIRA_WEB_OUTPUT_ROOT", APP_DIR / "persistent-output"))

os.environ["ENVIRA_WEB_OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ.setdefault("ENVIRA_WEB_TEMP_ROOT", "/content/envira-layout-web" if IN_COLAB else str(APP_DIR / "runtime"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
probe = OUTPUT_ROOT / ".envira-write-probe"
probe.write_text("ok\n", encoding="utf-8")
probe.unlink()
print("Persistent output root:", OUTPUT_ROOT)


In [ ]:
from envira_layout_web import AppSettings, create_app
from envira_layout_web.services.model_service import ModelService

settings = AppSettings.load(APP_DIR)
settings.prepare()

# If this setup cell is rerun after stopping the launch cell, close the old
# server first. Its temporary Gradio share URL expires when it is closed.
previous_demo = globals().get("demo")
if previous_demo is not None:
    previous_demo.close()

models = ModelService.initialize(settings)
demo = create_app(settings, models=models)
print("Models validated and Gradio application initialized.")


In [ ]:
# Ask Gradio to publish its normal public share URL rather than presenting
# Colab's kernel-local localhost URL as the application entry point.
demo.queue(
    default_concurrency_limit=settings.concurrency_limit,
).launch(
    share=True,
    inline=False,
    debug=IN_COLAB,
)
